# 06 — Domain-Specific Feature Engineering

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 50 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Create **ratio and difference features** that encode business logic directly
2. Build **aggregation features** from related entities (host-level statistics)
3. Compute **geographic features** using Haversine distance and spatial density
4. Design **domain KPIs** as features (occupancy proxy, experience premium)
5. Recognise and prevent **target leakage** in aggregation features

---

**Week 12 Theme:** Airbnb-style listing price prediction

## The Doctor Analogy

> *"A doctor does not use raw blood test numbers — they compute ratios like LDL/HDL cholesterol, or trends over time. In Airbnb pricing, 'price per bedroom' matters more than raw price; 'reviews per year' matters more than total reviews. Domain features embed causal understanding that generic transformations miss."*

Consider two Airbnb listings:

| Listing | Price | Bedrooms | Reviews | Host tenure |
|---------|-------|----------|---------|-------------|
| A | \$200 | 1 | 200 | 2 years |
| B | \$200 | 4 | 50  | 5 years |

They have the same raw price, but they are very different:
- Listing A: \$200/bedroom — **very expensive** studio in a prime location
- Listing B: \$50/bedroom — **good value** family home
- Listing A: 100 reviews/year — **extremely popular**, likely fully booked
- Listing B: 10 reviews/year — **niche/occasional** rental

These ratios are *domain knowledge made numeric*. A model cannot discover them from raw columns alone.

## Why Does This Matter for ML?

- **Top Kaggle competitors spend 60%+ of their time on domain features** — not on model tuning
- Domain features are **causally grounded**: `price_per_bedroom` is related to price *because* of how the market works, not just statistical correlation
- They are **robust across datasets**: a model trained with `reviews_per_year` will generalise better than one trained on raw `num_reviews`, because the denominator removes confounding from host tenure
- They often allow **simpler, more interpretable models** to match the performance of complex models on raw features

| Feature type | Example | Why it's better than raw |
|---|---|---|
| Ratio | `price / bedrooms` | Removes size confound |
| Rate | `reviews / years_hosting` | Removes tenure confound |
| Density | `listings within 1km` | Encodes competition |
| Aggregation | `host_mean_price` | Encodes host pricing strategy |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                          # reproducibility for all downstream cells
sns.set_theme(style='whitegrid')            # consistent seaborn theme

# ── Generate synthetic Airbnb listings dataset ───────────────────────────────
N = 2000                                    # total listings
N_HOSTS = 200                               # number of unique hosts

host_ids = np.random.randint(1, N_HOSTS + 1, size=N)

bedrooms   = np.random.choice([1, 1, 2, 2, 3, 4, 5], size=N)   # skewed toward smaller
bathrooms  = np.clip(bedrooms * np.random.uniform(0.5, 1.2, size=N), 1, 6).round(1)
accommodates = np.clip(bedrooms * np.random.randint(1, 3, size=N), 1, 16)

num_reviews       = np.random.poisson(lam=40, size=N).clip(0, 400)
review_score      = np.clip(np.random.normal(4.5, 0.4, size=N), 1.0, 5.0).round(2)
host_experience_days = np.random.randint(30, 3000, size=N)
is_superhost      = (host_experience_days > 500).astype(int) & \
                    (num_reviews > 20).astype(int)             # experience + reviews → superhost

# London bounding box: lat 51.3–51.7, lon -0.5–0.3
latitude  = np.random.uniform(51.35, 51.65, size=N)
longitude = np.random.uniform(-0.45, 0.25, size=N)

# Price model: base + bedroom premium + location (closer to centre = pricier) +
#              review score premium + superhost bump + noise
distance_to_centre = np.sqrt((latitude - 51.51)**2 + (longitude + 0.12)**2)
price = (
    50
    + bedrooms * 30
    + review_score * 12
    + is_superhost * 20
    - distance_to_centre * 80
    + np.random.normal(0, 20, size=N)
).clip(30, 500).round(2)

df = pd.DataFrame({
    'host_id':              host_ids,
    'bedrooms':             bedrooms,
    'bathrooms':            bathrooms,
    'accommodates':         accommodates,
    'num_reviews':          num_reviews,
    'review_score':         review_score,
    'host_experience_days': host_experience_days,
    'is_superhost':         is_superhost,
    'latitude':             latitude,
    'longitude':            longitude,
    'price':                price,
})

print(f'Dataset shape: {df.shape}')
print(f'Unique hosts: {df.host_id.nunique()}')
print(f'Price range: ${df.price.min():.0f} – ${df.price.max():.0f}')
df.describe().round(2)

## 1. Ratio Features

**Ratios normalise one variable by another** to remove a confound.

The key business ratios for Airbnb listings:

| Ratio | Formula | Business meaning |
|-------|---------|------------------|
| `price_per_bedroom` | price / bedrooms | Luxury level per room |
| `bathrooms_per_bedroom` | bathrooms / bedrooms | En-suite amenity density |
| `accommodates_per_bedroom` | accommodates / bedrooms | Occupancy density |
| `review_score_credibility` | score × log(reviews+1) | Credibility-weighted score |
| `reviews_per_year` | reviews / (experience_days/365) | Activity rate |

> **Tip:** always `.clip(lower=1)` denominators to avoid division-by-zero. Using `clip` is safer than `replace(0, 1)` because it handles floats and near-zero values too.

In [ ]:
np.random.seed(42)

# ── Compute ratio features with business justification ───────────────────────
df['price_per_bedroom'] = (
    df['price'] / df['bedrooms'].clip(lower=1)          # how much does each bedroom cost?
)
df['bathrooms_per_bedroom'] = (
    df['bathrooms'] / df['bedrooms'].clip(lower=1)      # en-suite density; >1 = luxury
)
df['accommodates_per_bedroom'] = (
    df['accommodates'] / df['bedrooms'].clip(lower=1)   # >3 = bunk beds / crowded
)
df['review_score_credibility'] = (
    df['review_score'] * np.log1p(df['num_reviews'])    # high score with few reviews is suspect
)
df['reviews_per_year'] = (
    df['num_reviews']
    / (df['host_experience_days'] / 365).clip(lower=0.1)  # annualised activity rate
)

# ── Heatmap: correlation of raw vs. ratio features with price ────────────────
raw_cols   = ['bedrooms', 'bathrooms', 'accommodates',
              'num_reviews', 'review_score', 'host_experience_days']
ratio_cols = ['price_per_bedroom', 'bathrooms_per_bedroom', 'accommodates_per_bedroom',
              'review_score_credibility', 'reviews_per_year']

raw_corr   = df[raw_cols   + ['price']].corr()['price'].drop('price')
ratio_corr = df[ratio_cols + ['price']].corr()['price'].drop('price')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw feature correlations
colors_raw = ['tomato' if v < 0 else 'steelblue' for v in raw_corr.values]
axes[0].barh(raw_corr.index, raw_corr.values, color=colors_raw)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Raw Feature Correlations\nwith Price', fontsize=11)
axes[0].set_xlabel('Pearson r')

# Right: ratio feature correlations
colors_ratio = ['tomato' if v < 0 else 'darkorange' for v in ratio_corr.values]
axes[1].barh(ratio_corr.index, ratio_corr.values, color=colors_ratio)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Ratio Feature Correlations\nwith Price', fontsize=11)
axes[1].set_xlabel('Pearson r')

plt.suptitle('Ratio features often have higher / more interpretable\ncorrelations with the target than raw features', y=1.02)
plt.tight_layout()
plt.show()

print('\nTop ratio features by |correlation| with price:')
print(ratio_corr.abs().sort_values(ascending=False).to_string())

## 2. Difference and Threshold Features

**Difference features** capture marginal effects — the value *beyond* a baseline:

- `bedrooms_beyond_1`: additional bedrooms beyond the minimum (1st bedroom is "standard", extras are premium)
- `accommodates_beyond_bedrooms`: how many guests sleep in living rooms / on sofas — signals a crammed listing

**Why not just use the raw column?** Because the relationship may be non-linear:
- Going from 0 to 1 bedroom → huge price jump
- Going from 4 to 5 bedrooms → much smaller marginal effect

Difference features encode this "diminishing returns" intuition directly.

In [ ]:
np.random.seed(42)

# ── Difference and threshold features ───────────────────────────────────────
df['bedrooms_beyond_1'] = (
    df['bedrooms'] - 1                      # extra bedrooms beyond first one
).clip(lower=0)                             # clip: a studio has 0, not -1

df['accommodates_beyond_bedrooms'] = (
    df['accommodates'] - df['bedrooms']     # guests in excess of one-per-bedroom
)                                           # high value = bunk beds / sofas

df['host_experience_months'] = (
    df['host_experience_days'] / 30         # months is more interpretable than days
)

# ── Analysis: overcrowded listings command a price discount ──────────────────
# Group accommodates_beyond_bedrooms into bins and plot mean price
df['crowd_bin'] = pd.cut(
    df['accommodates_beyond_bedrooms'],
    bins=[-1, 0, 1, 2, 3, 20],
    labels=['Same (0)', '+1 guest', '+2 guests', '+3 guests', '+4 or more']
)

crowd_price = df.groupby('crowd_bin', observed=True)['price'].agg(['mean', 'std', 'count'])

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    range(len(crowd_price)),
    crowd_price['mean'],
    color=['steelblue', 'steelblue', 'goldenrod', 'darkorange', 'tomato'],
    edgecolor='white'
)
# Error bars showing ± 1 std
ax.errorbar(range(len(crowd_price)), crowd_price['mean'],
            yerr=crowd_price['std'], fmt='none', color='black', capsize=4)
ax.set_xticks(range(len(crowd_price)))
ax.set_xticklabels(crowd_price.index, rotation=10)
ax.set_title('Mean Price by "Guests Beyond Bedrooms"\n'
             'Overcrowded listings trade off capacity for price', fontsize=11)
ax.set_xlabel('accommodates_beyond_bedrooms (binned)')
ax.set_ylabel('Mean price ($)')

# Annotate counts
for i, (idx, row) in enumerate(crowd_price.iterrows()):
    ax.text(i, row['mean'] + row['std'] + 2, f'n={int(row["count"])}',
            ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print('Insight: listings where guests >> bedrooms are cheaper (crowded feel).')
print('This signal is INVISIBLE in raw accommodates or bedrooms columns alone.')

## 3. Aggregation Features from Related Entities

Each listing belongs to a host. A host's **portfolio-level statistics** are powerful predictors of individual listing price:

- `host_mean_price`: a host who consistently charges \$300 almost certainly has a premium listing
- `host_num_listings`: hosts with many listings are likely professional operators with optimised pricing
- `host_std_price`: high variance = diverse portfolio; low variance = consistent strategy

The feature `price_vs_host_avg` (is this listing above or below the host's average?) is particularly useful — it captures whether this is the host's budget or premium offering.

In [ ]:
np.random.seed(42)

# ── Host-level aggregation features ─────────────────────────────────────────
host_stats = (
    df.groupby('host_id')['price']
    .agg(['mean', 'std', 'count'])          # mean, variability, portfolio size
    .rename(columns={
        'mean':  'host_mean_price',
        'std':   'host_std_price',
        'count': 'host_num_listings',
    })
)

# Fill NaN std (hosts with only 1 listing have no std)
host_stats['host_std_price'] = host_stats['host_std_price'].fillna(0)

# Merge back into listing-level DataFrame
df = df.merge(host_stats, on='host_id', how='left')

# How does this listing's price compare to the host's average?
df['price_vs_host_avg'] = (
    df['price'] / df['host_mean_price'].clip(lower=1)   # ratio > 1 = above host avg
)

# ── Visualise: host_mean_price vs individual listing price ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left scatter: host_mean vs listing price
axes[0].scatter(df['host_mean_price'], df['price'],
                alpha=0.3, s=15, color='steelblue')
# Perfect prediction line
lims = [df['host_mean_price'].min(), df['host_mean_price'].max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='y = x (perfect)')
corr_val = df['host_mean_price'].corr(df['price'])
axes[0].set_title(f'Host Mean Price vs Listing Price\nr = {corr_val:.3f}', fontsize=11)
axes[0].set_xlabel('host_mean_price')
axes[0].set_ylabel('listing price')
axes[0].legend()

# Right: distribution of host_num_listings — most hosts have 1 listing
listing_counts = host_stats['host_num_listings'].value_counts().sort_index()
axes[1].bar(listing_counts.index[:15], listing_counts.values[:15], color='darkorange')
axes[1].set_title('Host Portfolio Size Distribution\n(Most hosts have 1–3 listings)', fontsize=11)
axes[1].set_xlabel('Number of listings per host')
axes[1].set_ylabel('Number of hosts')

plt.tight_layout()
plt.show()

print('New host-level feature columns:')
print(df[['host_id', 'host_mean_price', 'host_std_price', 'host_num_listings', 'price_vs_host_avg']].head(8).to_string(index=False))

## 4. Geographic Features

Latitude and longitude are not useful as raw numbers — a model cannot easily learn that listings closer to the city centre command higher prices.

We transform them into **meaningful distances**:

### Haversine Distance

The straight-line distance between two points on Earth's surface:

$$d = 2R \arcsin\!\left(\sqrt{\sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos\phi_1 \cos\phi_2 \sin^2\!\left(\frac{\Delta\lambda}{2}\right)}\right)$$

where $R = 6371$ km, $\phi$ = latitude in radians, $\lambda$ = longitude in radians.

In plain English: it accounts for the curvature of the Earth when measuring distance — important for anything more than ~50km apart.

In [ ]:
np.random.seed(42)

def haversine(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in kilometres between two sets of coordinates.
    
    All inputs can be scalars or numpy arrays (vectorised).
    """
    R = 6371                                  # Earth's mean radius in km
    phi1 = np.radians(lat1)                   # convert degrees to radians
    phi2 = np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)         # difference in latitudes
    dlambda = np.radians(lon2 - lon1)         # difference in longitudes

    a = (np.sin(dphi / 2)**2
         + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2)

    return 2 * R * np.arcsin(np.sqrt(a))      # Haversine formula result in km

CENTRE_LAT, CENTRE_LON = 51.51, -0.12        # approximate centre of London

df['distance_km'] = haversine(
    df['latitude'], df['longitude'],
    CENTRE_LAT, CENTRE_LON
)

print(f'Distance range: {df.distance_km.min():.2f} – {df.distance_km.max():.2f} km')
print(f'Correlation with price: r = {df.distance_km.corr(df.price):.3f}')

# ── Scatter plot: price vs distance with regression line ─────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

# Colour by superhost status
for label, subset, color in [
    ('Regular host',   df[df['is_superhost'] == 0], 'steelblue'),
    ('Superhost',      df[df['is_superhost'] == 1], 'darkorange'),
]:
    ax.scatter(subset['distance_km'], subset['price'],
               alpha=0.3, s=12, color=color, label=label)

# Regression line (all data)
z = np.polyfit(df['distance_km'], df['price'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['distance_km'].min(), df['distance_km'].max(), 200)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (slope={z[0]:.1f} $/km)')

ax.set_title('Price vs Distance from City Centre\n'
             '(Closer to centre → higher price, even after controlling for size)', fontsize=11)
ax.set_xlabel('distance_km (from London centre)')
ax.set_ylabel('Listing price ($)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Listing density: how many competitors within 1km? ────────────────────────
from sklearn.neighbors import BallTree

# BallTree with haversine metric works directly on lat/lon in RADIANS
coords_rad = np.radians(df[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')

radius_km  = 1.0                             # 1 km search radius
radius_rad = radius_km / 6371                # convert km to radians for BallTree

# query_radius returns count of points within radius (including self → subtract 1)
counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
df['listings_within_1km'] = counts - 1      # exclude the listing itself

print(f'Density range: {df.listings_within_1km.min()} – {df.listings_within_1km.max()} nearby listings')
print(f'Correlation with price: r = {df.listings_within_1km.corr(df.price):.3f}')

# ── Scatter: density vs price ─────────────────────────────────────────────────
# Bin density for clearer signal
df['density_bin'] = pd.cut(
    df['listings_within_1km'],
    bins=[-1, 0, 5, 15, 30, 999],
    labels=['Isolated (0)', 'Low (1-5)', 'Medium (6-15)', 'High (16-30)', 'Very high (>30)']
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: map of listings coloured by density
sc = axes[0].scatter(df['longitude'], df['latitude'],
                     c=df['listings_within_1km'], cmap='YlOrRd',
                     s=8, alpha=0.6)
axes[0].scatter([CENTRE_LON], [CENTRE_LAT], marker='*', s=200,
                color='blue', zorder=5, label='City centre')
plt.colorbar(sc, ax=axes[0], label='Listings within 1km')
axes[0].set_title('Listing Density Map\n(Red = high competition)', fontsize=11)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend(fontsize=9)

# Right: price by density bin
density_price = df.groupby('density_bin', observed=True)['price'].mean()
density_price.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Mean Price by Listing Density\n'
                  '(High density = more competition = lower price)', fontsize=11)
axes[1].set_xlabel('Listings within 1km')
axes[1].set_ylabel('Mean price ($)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 5. Business KPI Features

Beyond simple ratios, experienced practitioners embed **domain KPIs** (Key Performance Indicators) as features. These are metrics the business already tracks — adapting them as model inputs gives the model the same knowledge a domain expert has.

For Airbnb, useful KPIs include:

| KPI Feature | Formula | What it captures |
|---|---|---|
| `occupancy_proxy` | reviews / (experience_months + 0.1) | How busy the listing is |
| `superhost_experience_premium` | is_superhost × log(experience_days+1) | Combined badge + tenure |
| `review_value_ratio` | review_score / log(price+1) | Quality per dollar |
| `capacity_efficiency` | accommodates / (bedrooms + bathrooms) | Space utilisation |

In [ ]:
np.random.seed(42)

# ── Airbnb-specific KPI features ────────────────────────────────────────────
df['occupancy_proxy'] = (
    df['num_reviews']
    / (df['host_experience_days'] / 30 + 0.1)           # monthly review rate (≈ occupancy)
)

df['superhost_experience_premium'] = (
    df['is_superhost'] * np.log1p(df['host_experience_days'])  # badge is worth more if earned slowly
)

df['review_value_ratio'] = (
    df['review_score'] / np.log1p(df['price'])           # high score at low price = great value
)

df['capacity_efficiency'] = (
    df['accommodates']
    / (df['bedrooms'] + df['bathrooms'] + 0.1)           # guests per physical room
)

# ── Feature importance from RandomForest ────────────────────────────────────
kpi_and_raw = [
    'bedrooms', 'bathrooms', 'accommodates', 'num_reviews',
    'review_score', 'host_experience_days', 'is_superhost', 'distance_km',
    'occupancy_proxy', 'superhost_experience_premium',
    'review_value_ratio', 'capacity_efficiency',
    'price_per_bedroom', 'bathrooms_per_bedroom', 'review_score_credibility',
    'reviews_per_year', 'listings_within_1km', 'host_mean_price',
]

X = df[kpi_and_raw].fillna(0).values
y = df['price'].values

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=kpi_and_raw).sort_values(ascending=True)

# Colour: orange = domain/KPI feature, blue = raw feature
domain_feats = {'occupancy_proxy', 'superhost_experience_premium', 'review_value_ratio',
                'capacity_efficiency', 'price_per_bedroom', 'bathrooms_per_bedroom',
                'review_score_credibility', 'reviews_per_year', 'host_mean_price',
                'distance_km', 'listings_within_1km'}
colors = ['darkorange' if f in domain_feats else 'steelblue' for f in importances.index]

fig, ax = plt.subplots(figsize=(9, 7))
importances.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importances — RandomForest (n=200 trees)\n'
             'Orange = domain / KPI features | Blue = raw features', fontsize=11)
ax.set_xlabel('Mean decrease in impurity (importance)')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
for feat, imp in importances.tail(5).iloc[::-1].items():
    tag = '[domain]' if feat in domain_feats else '[raw]'
    print(f'  {feat:<35} {imp:.4f}  {tag}')

In [ ]:
np.random.seed(42)

# ── Head-to-head: raw features only vs. raw + domain features ───────────────
raw_only = ['bedrooms', 'bathrooms', 'accommodates', 'num_reviews',
            'review_score', 'host_experience_days', 'is_superhost',
            'latitude', 'longitude']

all_feats = raw_only + [
    'price_per_bedroom', 'bathrooms_per_bedroom', 'accommodates_per_bedroom',
    'review_score_credibility', 'reviews_per_year',
    'bedrooms_beyond_1', 'accommodates_beyond_bedrooms',
    'host_mean_price', 'host_num_listings', 'price_vs_host_avg',
    'distance_km', 'listings_within_1km',
    'occupancy_proxy', 'superhost_experience_premium',
    'review_value_ratio', 'capacity_efficiency',
]

rf_eval = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

def r2_cv(X, y, cv=5):
    """Return mean and std of R² from 5-fold CV."""
    scores = cross_val_score(rf_eval, X, y, cv=cv, scoring='r2')
    return scores.mean(), scores.std()

r2_raw,  std_raw  = r2_cv(df[raw_only].fillna(0).values,  y)
r2_full, std_full = r2_cv(df[all_feats].fillna(0).values, y)

print('Cross-validated R² comparison:')
print(f'  Raw features only  ({len(raw_only):2d} features): R² = {r2_raw:.4f} ± {std_raw:.4f}')
print(f'  + Domain features  ({len(all_feats):2d} features): R² = {r2_full:.4f} ± {std_full:.4f}')
print(f'  Improvement:  +{r2_full - r2_raw:.4f} absolute R²')

# ── Bar chart summary ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
labels = [f'Raw only\n({len(raw_only)} features)', f'Raw + Domain\n({len(all_feats)} features)']
vals   = [r2_raw, r2_full]
errs   = [std_raw, std_full]
colors = ['steelblue', 'darkorange']
bars   = ax.bar(labels, vals, yerr=errs, color=colors, capsize=6, width=0.5, edgecolor='white')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Cross-validated R²')
ax.set_title('Domain Features Consistently Improve Model Performance', fontsize=11)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Leakage Risk in Domain Features

**Target leakage** is the most dangerous mistake in feature engineering. It happens when features are computed using information that would not be available at prediction time — usually because we use the *entire dataset* (including test data) when computing aggregations.

### The classic leakage scenario with aggregation features

Suppose we compute `host_mean_price` using all listings in the dataset, then use this feature to predict price. The test set rows are included in the aggregation — so the model effectively "sees" the test set labels during training. Cross-validation scores will be **inflated**.

**Safe pattern:** compute aggregation features using **only the training fold**, then apply them to the test/validation fold.

> This is sometimes called **Target Encoding done safely** — exactly the same principle applies when using mean-encoded categorical features.

In [ ]:
np.random.seed(42)

from sklearn.model_selection import KFold

# ── Simulate train/test split ────────────────────────────────────────────────
split_idx   = int(len(df) * 0.8)           # 80/20 split
train_df    = df.iloc[:split_idx].copy()
test_df     = df.iloc[split_idx:].copy()

# ── WRONG: compute host_mean_price on FULL dataset (leaky) ──────────────────
leaky_host_mean = df.groupby('host_id')['price'].mean()         # uses test prices too!
train_df['host_mean_leaky'] = train_df['host_id'].map(leaky_host_mean)
test_df['host_mean_leaky']  = test_df['host_id'].map(leaky_host_mean)

# ── CORRECT: compute host_mean_price on TRAINING data only ──────────────────
safe_host_mean  = train_df.groupby('host_id')['price'].mean()   # training data only
global_mean     = train_df['price'].mean()                      # fallback for unseen hosts

train_df['host_mean_safe'] = train_df['host_id'].map(safe_host_mean)
test_df['host_mean_safe']  = test_df['host_id'].map(safe_host_mean).fillna(global_mean)

# ── Evaluate the difference ──────────────────────────────────────────────────
from sklearn.metrics import r2_score

base_feats = ['bedrooms', 'bathrooms', 'accommodates', 'review_score', 'is_superhost']

rf_leak = RandomForestRegressor(n_estimators=100, random_state=42)
rf_safe = RandomForestRegressor(n_estimators=100, random_state=42)

X_train_leak = train_df[base_feats + ['host_mean_leaky']].fillna(0).values
X_test_leak  = test_df[base_feats  + ['host_mean_leaky']].fillna(0).values

X_train_safe = train_df[base_feats + ['host_mean_safe']].fillna(0).values
X_test_safe  = test_df[base_feats  + ['host_mean_safe']].fillna(0).values

y_train = train_df['price'].values
y_test  = test_df['price'].values

rf_leak.fit(X_train_leak, y_train)
rf_safe.fit(X_train_safe, y_train)

r2_leaky_train = r2_score(y_train, rf_leak.predict(X_train_leak))
r2_leaky_test  = r2_score(y_test,  rf_leak.predict(X_test_leak))

r2_safe_train  = r2_score(y_train, rf_safe.predict(X_train_safe))
r2_safe_test   = r2_score(y_test,  rf_safe.predict(X_test_safe))

print('Performance comparison (leaky vs safe host_mean_price):')
print(f'  Leaky — Train R²: {r2_leaky_train:.4f}   Test R²: {r2_leaky_test:.4f}')
print(f'  Safe  — Train R²: {r2_safe_train:.4f}   Test R²: {r2_safe_test:.4f}')
print(f'\n  Train-test gap (leaky): {r2_leaky_train - r2_leaky_test:.4f}  ← suspicious overfitting')
print(f'  Train-test gap (safe):  {r2_safe_train  - r2_safe_test:.4f}  ← realistic gap')

# ── Visual: train vs test R² comparison ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)                            # Train / Test
width = 0.3

ax.bar(x - width/2, [r2_leaky_train, r2_leaky_test], width,
       label='Leaky host_mean', color='tomato', edgecolor='white')
ax.bar(x + width/2, [r2_safe_train,  r2_safe_test],  width,
       label='Safe host_mean',  color='steelblue', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(['Train R²', 'Test R²'], fontsize=12)
ax.set_ylabel('R²')
ax.set_ylim(0, 1.05)
ax.set_title('Leaky Aggregation: High Train R² Drops on Test Set\n'
             'Safe Aggregation: Consistent Train / Test R²', fontsize=11)
ax.legend()

for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** You have a listing with `bedrooms = 0` (studio). Why do we use `.clip(lower=1)` instead of just filtering out these rows before computing `price_per_bedroom`?

<details><summary>Show answer</summary>

Filtering out studios would **remove valid data** — studios are real listings with real prices. `.clip(lower=1)` treats studios as having "1 bedroom-equivalent" for the purpose of the ratio, which is reasonable (a studio is the equivalent of a 1-bedroom unit in terms of capacity). This avoids division-by-zero while keeping all rows in the dataset.

</details>

---

**Q2.** Your colleague computes `host_mean_price` on the full dataset, uses it as a feature, and reports R² = 0.98 on 5-fold cross-validation. What is happening, and how would you fix it?

<details><summary>Show answer</summary>

This is **target leakage**. When `host_mean_price` is computed on the full dataset before splitting, the test fold rows are included in the mean. So when the model sees a test row, its `host_mean_price` partially reflects the test row's own `price` — giving the model indirect access to the answer. The fix: compute `host_mean_price` **only from the training fold** inside each cross-validation loop, then apply the trained mean mapping to the validation fold. Unseen hosts get the global training mean as a fallback.

</details>

---

**Q3.** The Haversine formula gives the great-circle distance. For listings in a city like London (spread over ~30km), would the simpler Euclidean distance `sqrt(Δlat² + Δlon²)` give noticeably different results?

<details><summary>Show answer</summary>

For small distances (< 50 km), the Euclidean approximation on raw lat/lon coordinates is **reasonably close** but not accurate for two reasons: (1) Earth's curvature introduces ~0.1–1% error at city scale; (2) a degree of longitude is *shorter* than a degree of latitude (cosine factor). A better approximation is the **equirectangular projection**: `d ≈ R * sqrt((Δlat)² + (cos(lat_mean) * Δlon)²)`. Haversine is still preferred in production because it is exact and the extra computation cost is negligible.

</details>

## Key Takeaways

1. **Ratios and rates are the most reliable domain features.** `price_per_bedroom` and `reviews_per_year` remove confounding variables that the raw columns carry, giving models a cleaner signal.

2. **Aggregation features (host-level stats) often top feature importance charts** — they encode the pricing strategy and quality level of the entity, which is highly predictive of individual listing behaviour.

3. **Geographic features must be transformed.** Raw latitude/longitude are nearly useless; `distance_km` and `listings_within_1km` are immediately interpretable and strongly predictive.

4. **Always check temporal logic and leakage** when computing aggregations. If your aggregation uses any information from the test set, your CV scores are lying. Compute group statistics from training data only, then map onto the validation/test set.

5. **Domain knowledge beats brute-force feature generation.** Generating 1000 polynomial combinations rarely beats 10 carefully chosen domain features — and the domain features are interpretable, which matters in production.

---

*Next up: **07 — Filter Methods for Feature Selection** — removing noise and redundancy from your feature set.*